# Retail Feature Engineering and Validation

This notebook follows the Springboard mini-project brief for retail weekly sales prediction. It loads the dataset from the Kaggle input path, engineers the required features, prevents data leakage during preprocessing, performs time-based splits, implements the Adstock and lag transformation functions, and writes submission-ready outputs.

## Deliverables Covered
- Processed datasets saved as CSV files
- Python notebook with explanations for every major step
- Generated summary markdown file for the one-page findings deliverable

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

DATA_PATH_CANDIDATES = [
    Path('/kaggle/input/datasets/mkishore129/retail-feature-engineering-assignment/retail_feature_engineering_assignment.csv'),
    Path('/kaggle/input/retail-feature-engineering-assignment/retail_feature_engineering_assignment.csv'),
    Path('retail_feature_engineering_assignment.csv'),
]
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = [
    'store_id',
    'week_start_date',
    'weekly_sales',
    'store_area',
    'num_employees',
    'store_rating',
    'holiday_flag',
    'temperature',
    'fuel_price',
    'cpi',
    'unemployment',
]
NUMERIC_COLUMNS = [
    'weekly_sales',
    'store_area',
    'num_employees',
    'store_rating',
    'temperature',
    'fuel_price',
    'cpi',
    'unemployment',
]
SCALE_COLUMNS = ['temperature', 'fuel_price', 'cpi', 'unemployment']
CORRELATION_THRESHOLD = 0.8
LEAKAGE_FEATURES = ['log_weekly_sales', 'sales_per_employee']

def resolve_data_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find the dataset in any of these locations: {candidates}')

DATA_PATH = resolve_data_path(DATA_PATH_CANDIDATES)
print(f'Using dataset: {DATA_PATH}')
print(f'Outputs will be written to: {OUTPUT_DIR.resolve()}')

In [ ]:
def validate_schema(df: pd.DataFrame) -> None:
    missing_columns = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

def bucket_store_rating(value):
    if pd.isna(value):
        return np.nan
    if value <= 3:
        return 'Low'
    if value > 7:
        return 'High'
    return 'Medium'

def fit_zscore(train_series: pd.Series) -> tuple[float, float]:
    mean_value = float(train_series.mean())
    std_value = float(train_series.std(ddof=0))
    if not np.isfinite(std_value) or std_value == 0:
        std_value = 1.0
    return mean_value, std_value

def apply_zscore(series: pd.Series, mean_value: float, std_value: float) -> pd.Series:
    return (series - mean_value) / std_value

def make_aligned_dummies(series: pd.Series, prefix: str, reference_columns=None) -> pd.DataFrame:
    dummies = pd.get_dummies(series, prefix=prefix, dtype='int64')
    if reference_columns is not None:
        dummies = dummies.reindex(columns=reference_columns, fill_value=0)
    return dummies

def find_high_correlation_features(df: pd.DataFrame, threshold: float = 0.8):
    corr_matrix = df.corr(numeric_only=True).abs()
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_pairs = []
    drop_features = []

    for column in upper_triangle.columns:
        correlated_rows = upper_triangle.index[upper_triangle[column] > threshold].tolist()
        for row_name in correlated_rows:
            corr_value = float(upper_triangle.loc[row_name, column])
            high_corr_pairs.append({
                'feature_1': row_name,
                'feature_2': column,
                'absolute_correlation': corr_value,
                'drop_recommendation': column,
            })
            if column not in drop_features:
                drop_features.append(column)

    high_corr_df = pd.DataFrame(
        high_corr_pairs,
        columns=['feature_1', 'feature_2', 'absolute_correlation', 'drop_recommendation'],
    )
    if not high_corr_df.empty:
        high_corr_df = high_corr_df.sort_values('absolute_correlation', ascending=False).reset_index(drop=True)
    return corr_matrix, high_corr_df, drop_features

def adstock_transformation(data: pd.Series, decay_rate: float, lags: int) -> pd.Series:
    if not isinstance(data, pd.Series):
        raise TypeError('data must be a pandas Series.')
    if data.empty:
        return pd.Series(dtype='float64', name=f'{data.name}_adstock')
    if not 0 <= decay_rate <= 1:
        raise ValueError('decay_rate must be between 0 and 1 inclusive.')
    if not isinstance(lags, int) or lags < 0:
        raise ValueError('lags must be a non-negative integer.')

    numeric_series = pd.to_numeric(data, errors='coerce')
    if numeric_series.isna().any():
        raise ValueError('data contains non-numeric values that cannot be transformed.')

    transformed_values = []
    for current_index in range(len(numeric_series)):
        effective_lags = min(lags, current_index)
        adstock_value = 0.0
        for lag in range(effective_lags + 1):
            adstock_value += (decay_rate ** lag) * numeric_series.iloc[current_index - lag]
        transformed_values.append(adstock_value)

    return pd.Series(transformed_values, index=data.index, name=f'{data.name}_adstock')

def lag_transformation(data: pd.Series, lags: list[int]) -> pd.DataFrame:
    if not isinstance(data, pd.Series):
        raise TypeError('data must be a pandas Series.')
    if lags is None:
        raise ValueError('lags cannot be None.')

    cleaned_lags = []
    for lag in lags:
        if not isinstance(lag, int) or lag <= 0:
            raise ValueError('Each lag must be a positive integer.')
        if lag not in cleaned_lags:
            cleaned_lags.append(lag)

    lagged_frame = pd.DataFrame(index=data.index)
    for lag in sorted(cleaned_lags):
        lagged_frame[f'Lag_{lag}'] = data.shift(lag)
    return lagged_frame

## 1. Load and Validate the Dataset
The dataset is validated against the expected schema, date parsing is checked explicitly, duplicates are removed, and core target requirements are enforced before feature engineering begins.

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
raw_df.columns = [column.strip() for column in raw_df.columns]
validate_schema(raw_df)

df = raw_df.copy()
df['week_start_date'] = pd.to_datetime(df['week_start_date'], dayfirst=True, errors='coerce')

for column in NUMERIC_COLUMNS + ['holiday_flag']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

invalid_date_rows = int(df['week_start_date'].isna().sum())
missing_target_rows = int(df['weekly_sales'].isna().sum())
duplicate_rows_removed = int(df.duplicated().sum())

if invalid_date_rows > 0 or missing_target_rows > 0:
    display(Markdown(
        f'**Rows removed during validation:** invalid dates = {invalid_date_rows}, missing target = {missing_target_rows}'
    ))

df = df.dropna(subset=['week_start_date', 'weekly_sales']).drop_duplicates().copy()
df = df.sort_values(['week_start_date', 'store_id']).reset_index(drop=True)

if (df['weekly_sales'] < 0).any():
    raise ValueError('weekly_sales contains negative values. log1p requires non-negative sales.')

holiday_values = set(df['holiday_flag'].dropna().astype(int).unique().tolist())
if not holiday_values.issubset({0, 1}):
    raise ValueError(f'holiday_flag contains values outside 0/1: {sorted(holiday_values)}')

quality_summary = pd.DataFrame([
    {
        'rows_after_validation': len(df),
        'stores': df['store_id'].nunique(),
        'start_date': df['week_start_date'].min().date(),
        'end_date': df['week_start_date'].max().date(),
        'duplicate_rows_removed': duplicate_rows_removed,
        'zero_employee_rows': int(df['num_employees'].fillna(0).eq(0).sum()),
    }
])

missing_summary = df.isna().sum().rename('missing_count').to_frame()
display(quality_summary)
display(missing_summary.T)
display(df.head())

## 2. Feature Engineering
Required engineered features are created here. The notebook also flags features that would leak target information if they were used as model inputs.

In [ ]:
df['log_weekly_sales'] = np.log1p(df['weekly_sales'])
df['sales_per_employee'] = np.where(
    df['num_employees'].fillna(0) > 0,
    df['weekly_sales'] / df['num_employees'],
    np.nan,
)
df['month'] = df['week_start_date'].dt.month.astype('Int64')
df['day'] = df['week_start_date'].dt.day.astype('Int64')
df['store_rating_category'] = df['store_rating'].apply(bucket_store_rating)

leakage_report = pd.DataFrame([
    {
        'feature': 'log_weekly_sales',
        'why_it_leaks': 'It is a deterministic transformation of the target weekly_sales.'
    },
    {
        'feature': 'sales_per_employee',
        'why_it_leaks': 'It directly uses weekly_sales in its numerator and would inject target information into model features.'
    },
])

display(leakage_report)
display(df[[
    'store_id',
    'week_start_date',
    'weekly_sales',
    'log_weekly_sales',
    'sales_per_employee',
    'month',
    'day',
    'store_rating_category',
]].head())

## 3. Temporal Splits, Leakage-Safe Preprocessing, and Collinearity Checks
The test set is defined as 2024 onward. The 2023 training window is then split chronologically into an 80% training subset and a 20% validation subset using unique weeks. All preprocessing steps that learn from data, including imputation, scaling, and one-hot category discovery, are fit on the training subset only.

In [ ]:
train_df = df[df['week_start_date'] < pd.Timestamp('2024-01-01')].copy()
test_df = df[df['week_start_date'] >= pd.Timestamp('2024-01-01')].copy()

if train_df.empty or test_df.empty:
    raise ValueError('Temporal split failed because either the train or test partition is empty.')

train_weeks = np.array(sorted(train_df['week_start_date'].unique()))
if len(train_weeks) < 2:
    raise ValueError('Not enough pre-2024 weeks to create an 80/20 time-based split.')

split_index = int(np.floor(len(train_weeks) * 0.8))
split_index = min(max(split_index, 1), len(train_weeks) - 1)
validation_start_week = train_weeks[split_index]

train_inner_df = train_df[train_df['week_start_date'] < validation_start_week].copy()
validation_df = train_df[train_df['week_start_date'] >= validation_start_week].copy()

continuous_impute_columns = [
    'store_area',
    'num_employees',
    'store_rating',
    'temperature',
    'fuel_price',
    'cpi',
    'unemployment',
    'month',
    'day',
]
impute_values = train_inner_df[continuous_impute_columns].median(numeric_only=True)

dataset_parts = {
    'train': train_inner_df,
    'validation': validation_df,
    'test': test_df,
}

for split_name, split_df in dataset_parts.items():
    split_df[continuous_impute_columns] = split_df[continuous_impute_columns].fillna(impute_values)
    split_df['holiday_flag'] = split_df['holiday_flag'].fillna(0).astype('Int64')
    split_df['store_rating_category'] = split_df['store_rating_category'].fillna('Unknown')

scaling_parameters = {}
for column in SCALE_COLUMNS:
    mean_value, std_value = fit_zscore(train_inner_df[column])
    scaling_parameters[column] = {'mean': mean_value, 'std': std_value}
    for split_df in dataset_parts.values():
        split_df[f'{column}_z'] = apply_zscore(split_df[column], mean_value, std_value)

holiday_columns = make_aligned_dummies(train_inner_df['holiday_flag'].astype('Int64'), 'holiday').columns.tolist()
month_columns = make_aligned_dummies(train_inner_df['month'].astype('Int64'), 'month').columns.tolist()
store_columns = make_aligned_dummies(train_inner_df['store_id'], 'store').columns.tolist()
rating_columns = make_aligned_dummies(train_inner_df['store_rating_category'], 'rating_cat').columns.tolist()

encoded_parts = {}
for split_name, split_df in dataset_parts.items():
    encoded_parts[split_name] = pd.concat(
        [
            split_df.reset_index(drop=True),
            make_aligned_dummies(split_df['holiday_flag'].astype('Int64'), 'holiday', holiday_columns).reset_index(drop=True),
            make_aligned_dummies(split_df['month'].astype('Int64'), 'month', month_columns).reset_index(drop=True),
            make_aligned_dummies(split_df['store_id'], 'store', store_columns).reset_index(drop=True),
            make_aligned_dummies(split_df['store_rating_category'], 'rating_cat', rating_columns).reset_index(drop=True),
        ],
        axis=1,
    )

correlation_features = [
    'store_area',
    'num_employees',
    'store_rating',
    'day',
    'temperature_z',
    'fuel_price_z',
    'cpi_z',
    'unemployment_z',
]
corr_matrix, high_corr_pairs, dropped_for_collinearity = find_high_correlation_features(
    encoded_parts['train'][correlation_features],
    threshold=CORRELATION_THRESHOLD,
)

if high_corr_pairs.empty:
    display(Markdown('**Collinearity result:** No continuous feature pairs exceeded the 0.8 threshold on the training split.'))
else:
    display(high_corr_pairs)

model_drop_columns = [
    'store_id',
    'holiday_flag',
    'month',
    'store_rating_category',
    'temperature',
    'fuel_price',
    'cpi',
    'unemployment',
] + LEAKAGE_FEATURES + dropped_for_collinearity

model_ready_parts = {}
for split_name, split_df in encoded_parts.items():
    model_part = split_df.copy()
    model_part['dataset_split'] = split_name
    model_part = model_part.drop(columns=[column for column in model_drop_columns if column in model_part.columns])
    model_ready_parts[split_name] = model_part

processed_full_df = pd.concat(
    [
        train_inner_df.assign(dataset_split='train'),
        validation_df.assign(dataset_split='validation'),
        test_df.assign(dataset_split='test'),
    ],
    axis=0,
).sort_values(['week_start_date', 'store_id']).reset_index(drop=True)

model_ready_df = pd.concat(model_ready_parts.values(), axis=0).sort_values(['week_start_date']).reset_index(drop=True)
feature_columns = [column for column in model_ready_parts['train'].columns if column not in ['weekly_sales', 'week_start_date', 'dataset_split']]

X_train = model_ready_parts['train'][feature_columns].copy()
y_train = model_ready_parts['train']['weekly_sales'].copy()
X_validation = model_ready_parts['validation'][feature_columns].copy()
y_validation = model_ready_parts['validation']['weekly_sales'].copy()
X_test = model_ready_parts['test'][feature_columns].copy()
y_test = model_ready_parts['test']['weekly_sales'].copy()

split_summary = pd.DataFrame([
    {
        'split': 'train',
        'rows': len(model_ready_parts['train']),
        'start': model_ready_parts['train']['week_start_date'].min().date(),
        'end': model_ready_parts['train']['week_start_date'].max().date(),
    },
    {
        'split': 'validation',
        'rows': len(model_ready_parts['validation']),
        'start': model_ready_parts['validation']['week_start_date'].min().date(),
        'end': model_ready_parts['validation']['week_start_date'].max().date(),
    },
    {
        'split': 'test',
        'rows': len(model_ready_parts['test']),
        'start': model_ready_parts['test']['week_start_date'].min().date(),
        'end': model_ready_parts['test']['week_start_date'].max().date(),
    },
])

display(split_summary)
display(pd.DataFrame(scaling_parameters).T)
display(pd.DataFrame({
    'feature_columns': feature_columns
}).head(20))
print('X_train shape:', X_train.shape)
print('X_validation shape:', X_validation.shape)
print('X_test shape:', X_test.shape)
print('Dropped for collinearity:', dropped_for_collinearity if dropped_for_collinearity else 'None')

full_output_path = OUTPUT_DIR / 'retail_processed_full.csv'
model_output_path = OUTPUT_DIR / 'retail_model_ready.csv'
processed_full_df.to_csv(full_output_path, index=False)
model_ready_df.to_csv(model_output_path, index=False)
print(f'Saved full processed dataset to: {full_output_path}')
print(f'Saved model-ready dataset to: {model_output_path}')

## 4. Adstock and Lag Transformations
The project brief also asks for two standalone transformation functions. The cells below test them on the provided toy datasets and interpret the first few results.

In [ ]:
ad_df = pd.DataFrame({
    'Date': pd.date_range(start='2023-01-01', periods=10, freq='D'),
    'Ad Spend': [1000, 1200, 1500, 1100, 1300, 1600, 1400, 1700, 1900, 1500],
})
ad_series = ad_df.set_index('Date')['Ad Spend']
adstock_series = adstock_transformation(ad_series, decay_rate=0.8, lags=3)

adstock_results = pd.concat([ad_series.rename('Ad Spend'), adstock_series.rename('Adstock_0.8_3')], axis=1)
display(adstock_results)

display(Markdown(
    'The first few Adstock values show carryover clearly: day 1 stays at 1000 because no history exists; day 2 becomes 2000 because it combines current spend of 1200 with 80% of the previous 1000; day 3 rises to 3100 because the last two days still contribute through exponential decay. This reflects marketing memory: older spend still matters, but each older period contributes less than the one before it.'
))

temp_df = pd.DataFrame({
    'Date': pd.date_range(start='2023-01-01', periods=10, freq='D'),
    'Temperature': [20, 22, 25, 23, 26, 28, 27, 29, 31, 30],
})
temp_series = temp_df.set_index('Date')['Temperature']
lagged_temperature = lag_transformation(temp_series, lags=[1, 2, 7])
display(pd.concat([temp_series.rename('Temperature'), lagged_temperature], axis=1))

display(Markdown(
    'Lagged features let a forecasting model learn temporal dependence. For example, `Lag_1` captures yesterdays temperature, `Lag_2` captures the short-term pattern from two days ago, and `Lag_7` can capture weekly seasonality. The initial rows are expected to contain missing values because those earlier observations do not yet exist.'
))

In [ ]:
summary_lines = [
    '# One-Page Summary',
    '',
    '## Dataset Overview',
    f'- Final validated row count: {len(df)}',
    f'- Number of stores: {df["store_id"].nunique()}',
    f'- Date coverage: {df["week_start_date"].min().date()} to {df["week_start_date"].max().date()}',
    '',
    '## Feature Engineering Completed',
    '- Created log_weekly_sales, sales_per_employee, month, day, and store_rating_category.',
    '- Applied train-only z-score scaling to temperature, fuel_price, cpi, and unemployment.',
    '- One-hot encoded holiday_flag, month, store_id, and store_rating_category using categories learned from the training split only.',
    '',
    '## Leakage and Validation Controls',
    '- Used a strict temporal split: all dates before 2024 for development and 2024 onward for testing.',
    '- Split the pre-2024 development data into an 80/20 chronological training and validation split using unique weeks.',
    '- Excluded log_weekly_sales and sales_per_employee from the model-ready feature set because they are derived from the target.',
    '',
    '## Collinearity Review',
    f'- Correlation threshold used: {CORRELATION_THRESHOLD}',
    f'- Features dropped for collinearity: {dropped_for_collinearity if dropped_for_collinearity else "None"}',
    '',
    '## Output Files',
    f'- Full processed dataset: {full_output_path.as_posix()}',
    f'- Model-ready dataset: {model_output_path.as_posix()}',
    '',
    '## Interpretation',
    '- The pipeline preserves time order, avoids fitting preprocessing steps on future data, and keeps a clean separation between analysis-only target-derived variables and legitimate model features.',
]

summary_text = '\n'.join(summary_lines)
summary_output_path = OUTPUT_DIR / 'submission_summary.md'
summary_output_path.write_text(summary_text, encoding='utf-8')
display(Markdown(summary_text))
print(f'Saved summary to: {summary_output_path}')